In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold ,cross_validate
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error 



from sklearn.linear_model import ( LinearRegression ,Ridge, Lasso , ElasticNet )
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor



from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ( RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor )


from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


from sklearn.ensemble import StackingRegressor, VotingRegressor


from sklearn.model_selection import GridSearchCV, RandomizedSearchCV



In [3]:
df = pd.read_csv("../data/processed/preprocessed.csv")

X = df.drop(["Global_active_power", "Date", "Time","Voltage"], axis=1)
y = df["Global_active_power"]

In [4]:
X

,Global_reactive_power,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.418,18.4,0.0,1.0,17.0
1,0.436,23.0,0.0,1.0,16.0
2,0.498,23.0,0.0,2.0,17.0
3,0.502,23.0,0.0,1.0,17.0
4,0.528,15.8,0.0,1.0,17.0
...,...,...,...,...,...
233,0.054,14.4,0.0,0.0,17.0
234,0.074,8.6,0.0,0.0,18.0
235,0.056,14.4,0.0,0.0,17.0
236,0.054,14.4,0.0,0.0,17.0


In [8]:
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
os.makedirs("../data/test data", exist_ok=True)


X_test.to_csv("../data/test data/X_test.csv", index=False)
y_test.to_csv("../data/test data/y_test.csv", index=False)

print("Test data saved successfully!")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Test data saved successfully!
X_test: (48, 5)
y_test: (48,)


In [10]:
kf = KFold( n_splits=3, shuffle=True, random_state=42)
print("3-Fold Cross Validation created.")

3-Fold Cross Validation created.


In [11]:
scaled_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "ElasticNet Regression": ElasticNet(),
    "KNN Regressor": KNeighborsRegressor(),
    "Support Vector Regressor": SVR()
}

tree_models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0)
}


In [12]:
# 6. Train, Evaluate via Cross-Validation & Store Results
trained_models = {}
results = []

all_models = {**scaled_models, **tree_models}

for name, model in all_models.items():
  # Create Pipeline based on model type (scaling required for linear/distance models)
  if name in scaled_models:
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
  else:
    pipeline = Pipeline([('model', model)])

  # Using Cross-Validation for robust evaluation to avoid overfitting
  cv_results = cross_validate(
      pipeline,
      X_dev,
      y_dev,
      cv=kf,
      scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'],
      n_jobs=-1,
  )

  # Calculate mean metrics across folds
  r2 = cv_results['test_r2'].mean()
  mae = -cv_results['test_neg_mean_absolute_error'].mean()
  rmse = -cv_results['test_neg_root_mean_squared_error'].mean()

  # Fit final pipeline on entire dev set for future predictions
  pipeline.fit(X_dev, y_dev)
  trained_models[name] = pipeline

  results.append({'Model': name, 'R2 Score': r2, 'MAE': mae, 'RMSE': rmse})
  print(f'Finished training and evaluating: {name}')

# 7. Compile and Display Results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='R2 Score', ascending=False).reset_index(
    drop=True
)

print('\n--- Final Model Performance (5-Fold CV) ---')
print(results_df)

Finished training and evaluating: Linear Regression
Finished training and evaluating: Ridge Regression
Finished training and evaluating: Lasso Regression
Finished training and evaluating: ElasticNet Regression
Finished training and evaluating: KNN Regressor
Finished training and evaluating: Support Vector Regressor
Finished training and evaluating: Decision Tree
Finished training and evaluating: Random Forest
Finished training and evaluating: Gradient Boosting
Finished training and evaluating: AdaBoost
Finished training and evaluating: XGBoost
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000105 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 76
[LightGBM] [Info] Number of data points in the train set: 190, number of used features: 4
[LightGBM] [Info] Start training from score 3.550632
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

c:\Users\vinay kumar\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
2 fits failed out of a total of 3.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\vinay kumar\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 856, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vinay kumar\anaconda3\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\vinay kumar\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 649, in fit
    self

Finished training and evaluating: CatBoost

--- Final Model Performance (5-Fold CV) ---
                       Model  R2 Score       MAE      RMSE
0          Linear Regression  0.996430  0.035641  0.053369
1           Ridge Regression  0.996400  0.036053  0.053666
2          Gradient Boosting  0.986725  0.050039  0.103752
3              Random Forest  0.985786  0.048892  0.105003
4                    XGBoost  0.983597  0.055871  0.115062
5                   AdaBoost  0.981103  0.075999  0.123578
6              Decision Tree  0.953645  0.071595  0.184244
7   Support Vector Regressor  0.871839  0.130904  0.319884
8              KNN Regressor  0.815789  0.197499  0.384725
9                   LightGBM  0.803155  0.241745  0.398631
10     ElasticNet Regression  0.485887  0.418212  0.645478
11          Lasso Regression -0.040902  0.598841  0.918415
12                  CatBoost       NaN       NaN       NaN


In [13]:
from sklearn.ensemble import StackingRegressor, VotingRegressor

# 1. Define the base estimators (Linear Regression aur Ridge Regression)
estimators = [
    ('linear', LinearRegression()),
    ('ridge', Ridge(alpha=1.0)),
]

# 2. Voting Regressor (Averages predictions of both models)
voting_regressor = VotingRegressor(estimators=estimators)

# 3. Stacking Regressor (Uses a final estimator, e.g., Ridge or LinearRegression, on top of base predictions)
# By default, final estimator is RidgeCV
stacking_regressor = StackingRegressor(
    estimators=estimators, final_estimator=Ridge()
)

ensemble_models = {
    'Voting Regressor': voting_regressor,
    'Stacking Regressor': stacking_regressor,
}

ensemble_results = []

# 4. Train and Evaluate using Cross-Validation
for name, model in ensemble_models.items():
  # Pipeline banakar scale karna zaroori hai kyunki Linear & Ridge models hain
  pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])

  cv_results = cross_validate(
      pipeline,
      X_dev,
      y_dev,
      cv=kf,
      scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'],
      n_jobs=-1,
  )

  r2 = cv_results['test_r2'].mean()
  mae = -cv_results['test_neg_mean_absolute_error'].mean()
  rmse = -cv_results['test_neg_root_mean_squared_error'].mean()

  ensemble_results.append({'Model': name, 'R2 Score': r2, 'MAE': mae, 'RMSE': rmse})
  print(f'Finished training and evaluating: {name}')

# 5. Display Results
ensemble_df = pd.DataFrame(ensemble_results)
print('\n--- Ensemble Models Performance ---')
print(ensemble_df)

Finished training and evaluating: Voting Regressor
Finished training and evaluating: Stacking Regressor

--- Ensemble Models Performance ---
                Model  R2 Score       MAE      RMSE
0    Voting Regressor  0.996444  0.035608  0.053280
1  Stacking Regressor  0.996398  0.035634  0.053647


In [20]:
from sklearn.model_selection import GridSearchCV 

# 1. Base models define karein
estimators = [('linear', LinearRegression()), ('ridge', Ridge())]

# 2. Voting Regressor banayein
voting_regressor = VotingRegressor(estimators=estimators)

# 3. Pipeline banayein (Scaler ke sath)
pipeline = Pipeline([('scaler', StandardScaler()), ('model', voting_regressor)])

# 4. Param Grid set karein (Yahan hum Ridge ke alpha ko tune kar rahe hain)
# Format: 'model__<estimator_name>__<parameter_name>'
param_grid = {'model__ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}

# 5. GridSearchCV setup karein
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
)

# 6. Fit karein (Chote subset ya poore data par)
grid_search.fit(X_dev, y_dev)

# 7. Best parameters aur best score print karein
print('Grid Search - Best Parameters:', grid_search.best_params_)
print('Grid Search - Best R2 Score:', grid_search.best_score_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Grid Search - Best Parameters: {'model__ridge__alpha': 1.0}
Grid Search - Best R2 Score: 0.9964444504295219


In [19]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Base models define karein
estimators = [('linear', LinearRegression()), ('ridge', Ridge())]

# 2. Voting Regressor banayein
voting_regressor = VotingRegressor(estimators=estimators)

# 3. Pipeline banayein
pipeline = Pipeline([('scaler', StandardScaler()), ('model', voting_regressor)])

# 4. Param Distributions (Ridge ke alpha ke liye alag-alag range)
param_distributions = {
    'model__ridge__alpha': [
        0.0001,
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
        1000.0,
    ]
}

# 5. RandomizedSearchCV setup karein
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=5,  # Kitni random combinations try karni hain
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

# 6. Fit karein
random_search.fit(X_dev, y_dev)

# 7. Best parameters aur score print karein
print('Best Parameters:', random_search.best_params_)
print('Best R2 Score:', random_search.best_score_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best Parameters: {'model__ridge__alpha': 0.01}
Best R2 Score: 0.9964300123385358


In [22]:
import os
import joblib  # joblib pandas aur scikit-learn ke sath models save karne ke liye sabse best aur fast hai

# 1. Models folder create karein agar pehle se nahi hai
os.makedirs('models', exist_ok=True)

# 2. GridSearchCV ya RandomizedSearchCV ke best estimator ko pick karein
# (Aapne upar grid search run kiya hai, toh grid_search.best_estimator_ ka use karenge)
best_model = grid_search.best_estimator_

# 3. Model ko pkl file mein save karein
model_path = '../models/best_voting_regressor.pkl'
joblib.dump(best_model, model_path)

print(f'Model successfully saved at: {model_path}')

Model successfully saved at: ../models/best_voting_regressor.pkl
